In [1]:
!pip install kagglehub tensorflow numpy

In [2]:
import kagglehub
import os
import numpy as np
import string
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [3]:
path = kagglehub.dataset_download("ashishpandey2062/next-word-predictor-text-generator-dataset")
print("Path:", path)

print(os.listdir(path))

100%|██████████| 61.5k/61.5k [00:00<00:00, 51.7MB/s]

Extracting files...
Path: /root/.cache/kagglehub/datasets/ashishpandey2062/next-word-predictor-text-generator-dataset/versions/1
['next_word_predictor.txt']


In [4]:
file_path = os.path.join(path, "next_word_predictor.txt")

with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read()

print(text[:500])

The sun was shining brightly in the clear blue sky, and a gentle breeze rustled the leaves of the tall trees. People were out enjoying the beautiful weather, some sitting in the park, others taking a leisurely stroll along the riverbank. Children were playing games, and laughter filled the air.

As the day turned into evening, the temperature started to drop, and the sky transformed into a canvas of vibrant colors. Families gathered for picnics, and the smell of barbecues wafted through the air.


In [5]:
import string

text = text.lower()
text = text.translate(str.maketrans('', '', string.punctuation))

In [6]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1
print("Total words:", total_words)

Total words: 5012


In [7]:
input_sequences = []

for line in text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

In [8]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_seq_len = max(len(seq) for seq in input_sequences)

input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')
print("Max sequence length:", max_seq_len)

Max sequence length: 322


In [9]:
import tensorflow as tf

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = tf.keras.utils.to_categorical(y, num_classes=total_words)

In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

model = Sequential()
model.add(Embedding(total_words, 100, input_length=max_seq_len-1))
model.add(SimpleRNN(150))
model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.fit(X, y, epochs=50, verbose=1)

Epoch 1/50
816/816 ━━━━━━━━━━━━━━━━━━━━ 28s 29ms/step - accuracy: 0.0649 - loss: 7.0211
Epoch 2/50
816/816 ━━━━━━━━━━━━━━━━━━━━ 21s 26ms/step - accuracy: 0.0933 - loss: 6.1577
Epoch 3/50
816/816 ━━━━━━━━━━━━━━━━━━━━ 21s 26ms/step - accuracy: 0.1152 - loss: 5.6377
Epoch 4/50
816/816 ━━━━━━━━━━━━━━━━━━━━ 21s 26ms/step - accuracy: 0.1464 - loss: 5.0477
Epoch 5/50
816/816 ━━━━━━━━━━━━━━━━━━━━ 21s 26ms/step - accuracy: 0.1828 - loss: 4.5205
Epoch 6/50
816/816 ━━━━━━━━━━━━━━━━━━━━ 21s 26ms/step - accuracy: 0.2381 - loss: 4.0299
Epoch 7/50
816/816 ━━━━━━━━━━━━━━━━━━━━ 21s 26ms/step - accuracy: 0.3100 - loss: 3.5682
Epoch 8/50
816/816 ━━━━━━━━━━━━━━━━━━━━ 21s 26ms/step - accuracy: 0.3840 - loss: 3.1460
Epoch 9/50
816/816 ━━━━━━━━━━━━━━━━━━━━ 21s 26ms/step - accuracy: 0.4557 - loss: 2.7633
Epoch 10/50
816/816 ━━━━━━━━━━━━━━━━━━━━ 21s 26ms/step - accuracy: 0.5152 - loss: 2.4372
Epoch 11/50
816/816 ━━━━━━━━━━━━━━━━━━━━ 21s 26ms/step - accuracy: 0.5637 - loss: 2.1739
Epoch 12/50
816/816 ━━━━━━━━━━

In [12]:
import numpy as np

def generate_text(seed_text, next_words):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')

        predicted = model.predict(token_list, verbose=0)
        predicted_word_index = np.argmax(predicted)

        for word, index in tokenizer.word_index.items():
            if index == predicted_word_index:
                seed_text += " " + word
                break

    return seed_text

In [14]:
print(generate_text("Nights in the village", 5))
print(generate_text("Adventurers from around", 5))

Nights in the village were enchanting with the stars
Adventurers from around the world journeyed to iceland
